# BTCPredictor2 - GPU Training

## Before running
1. Enable GPU: **Runtime -> Change runtime type -> A100 GPU** (recommended)
2. Run Cell 1 (clones repo + installs packages)
3. Run Cell 2 (mounts Drive and links yearly feature files)
4. Run Cells 3, 4, 5 to train TFT, BiLSTM, and Meta

## What to upload to Google Drive
Upload the entire `data/yearly/` folder from your local BTCPredictor2 to your Drive.
It contains: `2022_merged.csv`, `2023_merged.csv`, `2024_merged.csv`, `2025_merged.csv`, `2026_merged.csv`

**Estimated time on A100 GPU:**
- TFT: 2-4 hours
- BiLSTM: 30-60 minutes
- Meta: 5 minutes

In [ ]:
# ============================================================
# CELL 1 - SETUP
# Always deletes and re-clones so you get the latest GitHub push.
# ============================================================

GITHUB_URL = 'https://github.com/chefo919/BTCPredictor2.git'

import shutil, os
if os.path.exists('/content/BTCPredictor2'):
    shutil.rmtree('/content/BTCPredictor2')
    print('Removed old clone.')

!git clone {GITHUB_URL} /content/BTCPredictor2
!cd /content/BTCPredictor2 && git log --oneline -3

import sys
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

!pip install -q xgboost joblib ta scikit-learn-intelex

os.makedirs('/content/BTCPredictor2/data/yearly', exist_ok=True)
os.makedirs('/content/BTCPredictor2/models/saved', exist_ok=True)

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print(f'GPU: {gpus[0].name}  |  Mixed precision: ON')
else:
    print('WARNING: No GPU — Runtime > Change runtime type > GPU')

import config
config.BATCH_TFT    = 512
config.BATCH_BILSTM = 1024

from features.engineer import get_feature_groups
from models import tft_model
groups = get_feature_groups()

days = config.SEQ_LEN_TFT * tft_model.DOWNSAMPLE // (60 * 24)
print()
print(f'BiLSTM features: {len(groups["bilstm"])}  (1m/15m/30m — {config.SEQ_LEN_BILSTM}min minute-resolution context)')
print(f'TFT    features: {len(groups["tft_dynamic"])}  (h1/h4/d1 — {config.SEQ_LEN_TFT} steps x {tft_model.DOWNSAMPLE}min = {days} days context)')
print()
print(f'TFT:    SEQ={config.SEQ_LEN_TFT} x {tft_model.DOWNSAMPLE}min/step | HORIZON={config.HORIZON_TFT}min | batch={config.BATCH_TFT}')
print(f'BiLSTM: SEQ={config.SEQ_LEN_BILSTM} x 1min/step  | HORIZON={config.HORIZON_BILSTM}min | batch={config.BATCH_BILSTM}')
print()
print('Done. Run Cell 2 to link the training data.')

In [ ]:
# ============================================================
# CELL 2 - MOUNT DRIVE AND LINK YEARLY DATA FILES
# Upload your local data/yearly/ folder to Google Drive first.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, shutil

DEST_DIR = '/content/BTCPredictor2/data/yearly'
os.makedirs(DEST_DIR, exist_ok=True)

# Look for yearly files in common Drive locations
YEARLY_FILES = ['2022_merged.csv', '2023_merged.csv', '2024_merged.csv',
                '2025_merged.csv', '2026_merged.csv']

search_dirs = [
    '/content/drive/MyDrive/BTCPredictor2/data/yearly',
    '/content/drive/MyDrive/BTCPredictor2/yearly',
    '/content/drive/MyDrive/yearly',
    '/content/drive/MyDrive',
]

found_dir = None
for d in search_dirs:
    if os.path.exists(d) and any(os.path.exists(os.path.join(d, f)) for f in YEARLY_FILES):
        found_dir = d
        break

if found_dir is None:
    print('ERROR: Could not find yearly CSV files in Drive.')
    print('Please upload your local data/yearly/ folder to Google Drive.')
    print('Checked locations:')
    for d in search_dirs:
        print(f'  {d}')
else:
    linked = []
    for fname in YEARLY_FILES:
        src = os.path.join(found_dir, fname)
        dst = os.path.join(DEST_DIR, fname)
        if os.path.exists(src):
            if not os.path.exists(dst):
                os.symlink(src, dst)
            sz = os.path.getsize(src) / 1024**2
            import pandas as _pd
            rc = sum(1 for _ in open(src)) - 1
            print(f'  {fname}: {rc:,} rows  ({sz:.0f} MB)')
            linked.append(fname)
        else:
            print(f'  {fname}: NOT FOUND (skipping)')
    print()
    print(f'Linked {len(linked)} yearly files from {found_dir}')
    print('Ready. Run Cell 3 to train TFT.')

In [ ]:
# ============================================================
# CELL 3 - TRAIN TFT  (estimated: 30-60 min on A100)
# TFT uses 168-step hourly sequences = 7 days of macro context.
# Each step = 1 hour of real time -> attention sees genuine daily trends.
# batch=512 is safe: attention [512,4,168,168] = 55 MB float16.
# ============================================================

import os, sys, time
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

import tensorflow as tf
tf.keras.mixed_precision.set_global_policy('mixed_float16')

import config
config.BATCH_TFT = 512

import pandas as pd
from features.engineer import get_feature_groups
from models import tft_model

groups           = get_feature_groups()
TFT_DYN_FEATURES = groups['tft_dynamic']   # h1/h4/d1 -- 36 features
TFT_STA_FEATURES = groups['tft_static']    # [] -- no static covariates

days = config.SEQ_LEN_TFT * tft_model.DOWNSAMPLE // (60 * 24)
print(f'TFT dynamic features: {len(TFT_DYN_FEATURES)}  (h1/h4/d1)')
print(f'TFT sequence: {config.SEQ_LEN_TFT} steps x {tft_model.DOWNSAMPLE}min/step = {days} days')
print()

YEARLY_DIR  = 'data/yearly'
START_DATE  = '2022-02-01'
CUTOFF_DATE = config.TRAINING_CUTOFF_DATE
start_ts    = pd.Timestamp(START_DATE,  tz='UTC')
cutoff_ts   = pd.Timestamp(CUTOFF_DATE, tz='UTC')

dfs = []
for fname in sorted(os.listdir(YEARLY_DIR)):
    if not fname.endswith('_merged.csv'):
        continue
    df_y = pd.read_csv(os.path.join(YEARLY_DIR, fname), parse_dates=['time'])
    if df_y['time'].dt.tz is None:
        df_y['time'] = pd.to_datetime(df_y['time'], utc=True)
    df_y = df_y[(df_y['time'] >= start_ts) & (df_y['time'] <= cutoff_ts)]
    if not df_y.empty:
        dfs.append(df_y)
df = pd.concat(dfs, ignore_index=True).sort_values('time').reset_index(drop=True)
print(f'Rows (1m): {len(df):,}  ->  ~{len(df)//tft_model.DOWNSAMPLE:,} hourly rows after downsampling')
print(f'Training TFT  SEQ={tft_model.SEQ_LEN}h  HORIZON={tft_model.HORIZON}min  batch={config.BATCH_TFT}')
print()

t0 = time.time()
results = tft_model.train(df, TFT_DYN_FEATURES, TFT_STA_FEATURES)
elapsed = time.time() - t0

print()
print('TFT COMPLETE')
print(f'  Test accuracy: {results["test_acc"]:.4f}')
print(f'  Val  accuracy: {results["val_acc"]:.4f}')
print(f'  Time:          {int(elapsed//3600)}h {int((elapsed%3600)//60)}m')
print()
print('Run Cell 4 to train BiLSTM.')

In [ ]:
# ============================================================
# CELL 4 - TRAIN BILSTM  (estimated: 30-60 min on A100)
# ============================================================

import os, sys, time
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

import tensorflow as tf
tf.keras.mixed_precision.set_global_policy('mixed_float16')

import config
config.BATCH_BILSTM = 1024

import pandas as pd
from features.engineer import get_feature_groups
from models import bilstm_model

groups          = get_feature_groups()
BILSTM_FEATURES = groups['bilstm']   # 1m/15m/30m — 36 short-term features

print(f'BiLSTM features: {len(BILSTM_FEATURES)}  (1m/15m/30m — 12 indicators each)')
print()

# Load training data from yearly files
YEARLY_DIR  = 'data/yearly'
START_DATE  = '2022-02-01'
CUTOFF_DATE = config.TRAINING_CUTOFF_DATE
start_ts    = pd.Timestamp(START_DATE,  tz='UTC')
cutoff_ts   = pd.Timestamp(CUTOFF_DATE, tz='UTC')

dfs = []
for fname in sorted(os.listdir(YEARLY_DIR)):
    if not fname.endswith('_merged.csv'):
        continue
    df_y = pd.read_csv(os.path.join(YEARLY_DIR, fname), parse_dates=['time'])
    if df_y['time'].dt.tz is None:
        df_y['time'] = pd.to_datetime(df_y['time'], utc=True)
    df_y = df_y[(df_y['time'] >= start_ts) & (df_y['time'] <= cutoff_ts)]
    if not df_y.empty:
        dfs.append(df_y)
df = pd.concat(dfs, ignore_index=True).sort_values('time').reset_index(drop=True)
print(f'Rows: {len(df):,}  ({START_DATE} -> {CUTOFF_DATE})')
print(f'Training BiLSTM  SEQ_LEN={bilstm_model.SEQ_LEN}  HORIZON={bilstm_model.HORIZON}min')
print()

t0 = time.time()
results = bilstm_model.train(df, BILSTM_FEATURES)
elapsed = time.time() - t0

print()
print('BiLSTM COMPLETE')
print(f'  Test accuracy: {results["test_acc"]:.4f}')
print(f'  Val  accuracy: {results["val_acc"]:.4f}')
print(f'  Time:          {int(elapsed//3600)}h {int((elapsed%3600)//60)}m')
print()
print('Run Cell 5 to train Meta and download models.')

In [ ]:
# ============================================================
# CELL 5 - TRAIN META + DOWNLOAD  (~5 minutes)
#
# OOF timeline:
#   [0-70% train] [+24h gap] [72-90% OOF] [+24h gap] [92-100% test]
#
# 24h purge gaps prevent training labels from referencing OOF prices.
# XGBoost uses TimeSeriesSplit(3) CV internally -- no future leakage.
# ============================================================

import os, sys, time, shutil, json
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

import numpy as np
import pandas as pd
import config
from features.engineer import get_feature_groups
from models import tft_model, bilstm_model, meta_model

groups           = get_feature_groups()
BILSTM_FEATURES  = groups['bilstm']
TFT_DYN_FEATURES = groups['tft_dynamic']
TFT_STA_FEATURES = groups['tft_static']
ALL_NEEDED       = BILSTM_FEATURES + TFT_DYN_FEATURES + TFT_STA_FEATURES

# Load data
YEARLY_DIR  = 'data/yearly'
START_DATE  = '2022-02-01'
CUTOFF_DATE = config.TRAINING_CUTOFF_DATE
start_ts    = pd.Timestamp(START_DATE,  tz='UTC')
cutoff_ts   = pd.Timestamp(CUTOFF_DATE, tz='UTC')

dfs = []
for fname in sorted(os.listdir(YEARLY_DIR)):
    if not fname.endswith('_merged.csv'):
        continue
    df_y = pd.read_csv(os.path.join(YEARLY_DIR, fname), parse_dates=['time'])
    if df_y['time'].dt.tz is None:
        df_y['time'] = pd.to_datetime(df_y['time'], utc=True)
    df_y = df_y[(df_y['time'] >= start_ts) & (df_y['time'] <= cutoff_ts)]
    if not df_y.empty:
        dfs.append(df_y)
df = pd.concat(dfs, ignore_index=True).sort_values('time').reset_index(drop=True)
df = df.dropna(subset=ALL_NEEDED).reset_index(drop=True)

# OOF split with 24-hour purge gaps (same as train.py)
PURGE      = 24 * 60          # 24h in 1-minute rows
n_total    = len(df)
train_end  = int(n_total * 0.70)
oof_start  = train_end  + PURGE
oof_end    = int(n_total * 0.90)
test_start = oof_end    + PURGE

df_oof  = df.iloc[oof_start:oof_end].copy()
df_test = df.iloc[test_start:].copy()

print(f'Total rows:    {n_total:,}')
print(f'Train:         0 - {train_end:,}  (70%)')
print(f'OOF (meta):    {oof_start:,} - {oof_end:,}  ({len(df_oof):,} rows, 24h purge each side)')
print(f'Test:          {test_start:,} - end  ({len(df_test):,} rows)')
print()

val_X_dyn = df_oof[TFT_DYN_FEATURES].values.astype('float32')
val_X_sta = df_oof[TFT_STA_FEATURES].values.astype('float32')
val_X_bi  = df_oof[BILSTM_FEATURES].values.astype('float32')

acc_path   = 'models/saved/model_accuracies.json'
saved_acc  = json.load(open(acc_path)) if os.path.exists(acc_path) else {}
tft_val_err    = 1.0 - saved_acc.get('tft_val', 0.51)
bilstm_val_err = 1.0 - saved_acc.get('bilstm_val', 0.51)

print('Generating OOF predictions...')
val_tft_probs    = tft_model.predict_proba_batch(val_X_dyn, val_X_sta)
val_bilstm_probs = bilstm_model.predict_proba_batch(val_X_bi)

print('Training XGBoost meta-learner with TimeSeriesSplit CV...')
meta_results = meta_model.train(df_oof, val_tft_probs, val_bilstm_probs,
                                 tft_val_err, bilstm_val_err,
                                 TFT_DYN_FEATURES + TFT_STA_FEATURES)
w = meta_results.get('weights', {})

print()
print('=' * 50)
print('TRAINING COMPLETE')
print(f'  TFT accuracy:    {saved_acc.get("tft", 0):.3f}')
print(f'  BiLSTM accuracy: {saved_acc.get("bilstm", 0):.3f}')
print(f'  Meta CV acc:     {meta_results["train_acc"]:.3f}  (TimeSeriesSplit 3-fold)')
print(f'  TFT weight:      {w.get("tft", 0):.3f}')
print(f'  BiLSTM weight:   {w.get("bilstm", 0):.3f}')
print(f'  Data window:     {START_DATE} -> {CUTOFF_DATE}')
print('=' * 50)

# Zip and download
OUT = '/content/models_output'
os.makedirs(OUT, exist_ok=True)
for f in ['tft.keras', 'tft_scaler.pkl', 'tft_n_static.txt',
          'bilstm.keras', 'bilstm_scaler.pkl',
          'meta_xgb.pkl', 'model_accuracies.json', 'training_cutoff.txt']:
    src = f'models/saved/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{OUT}/{f}')

shutil.make_archive('/content/btc_models', 'zip', OUT)

from google.colab import files
print()
print('Downloading btc_models.zip to your PC...')
files.download('/content/btc_models.zip')
print()
print('Extract the zip and copy all files into your local models/saved/ folder.')
print('Then run: python papertrading/backtest.py --start 2026-04-15')